# NB07 블록과 체인 — 변조가 비싸지는 구조

『법률가를 위한 블록체인 — 기술과 경제학으로 읽기』의 장별 노트북입니다. 본문의 코드와 같습니다.

- 모든 코드는 원리를 눈으로 보기 위한 **장난감 구현**입니다. 실제 자산·실제 개인키·실제 거래에 쓰지 마십시오(실사용 금지).
- 외부 API를 부르지 않습니다. 네트워크 없이 실행됩니다. 동봉 스냅샷 데이터가 필요한 셀은 그 데이터를 셀 안에 적어 두었습니다.
- 표준 라이브러리만 씁니다(Python 3.13에서 실행·확인). 설치 셀은 없습니다.
- 출력은 엮은이가 실행한 결과를 그대로 담아 두었습니다. 다시 실행하면 같은 값이 나와야 합니다(난수는 시드를 고정했습니다).

In [ ]:
# NB07-1: 장난감 블록체인 — 작업증명 없는 해시 사슬. 변조는 드러나지만 다시 꿰는 것은 공짜다 (공부용)
import hashlib, json

def H(s: str) -> str:
    return hashlib.sha256(s.encode()).hexdigest()

class Block:
    def __init__(self, index, prev_hash, data, timestamp):
        self.index, self.prev_hash, self.data, self.timestamp = index, prev_hash, data, timestamp
    def header(self) -> str:                      # 헤더 = 사슬에 들어가는 부분
        return json.dumps([self.index, self.prev_hash, H(self.data), self.timestamp])
    def hash(self) -> str:                        # 블록 해시 = 헤더의 해시 (데이터는 H(data)로만 들어간다 — 머클 루트 자리)
        return H(self.header())

def build_chain(datas):
    chain = [Block(0, "0" * 64, datas[0], 1231006505)]            # 제네시스
    for i, d in enumerate(datas[1:], start=1):
        chain.append(Block(i, chain[-1].hash(), d, 1231006505 + 600 * i))
    return chain

def validate(chain):
    """각 블록의 prev_hash 가 실제 직전 블록 해시와 같은지 검사. 틀린 첫 블록 번호를 돌려준다(없으면 None)."""
    for i in range(1, len(chain)):
        if chain[i].prev_hash != chain[i - 1].hash():
            return i
    return None

datas = ["genesis", "A→B 1", "B→C 2", "C→D 3", "D→E 4", "E→F 5"]
chain = build_chain(datas)
print("블록 수:", len(chain), "| 유효?", validate(chain) is None)
for b in chain:
    print(f"  #{b.index} prev={b.prev_hash[:10]}… hash={b.hash()[:10]}… data={b.data!r}")

print("\n--- 3번 블록의 거래를 '3' 에서 '300' 으로 바꾼다 ---")
chain[3].data = "C→D 300"
bad = validate(chain)
print("처음으로 끊어진 자리: 블록 #", bad, "(3번을 바꿨는데 4번이 틀렸다고 나온다 — 4번의 prev_hash 가 옛 해시)")
for b in chain[3:]:
    print(f"  #{b.index} prev={b.prev_hash[:10]}… 실제 직전 해시={chain[b.index-1].hash()[:10]}…  일치={b.prev_hash == chain[b.index-1].hash()}")

print("\n--- 변조자가 이후 블록의 prev_hash 를 전부 다시 적는다 ---")
for i in range(4, len(chain)):
    chain[i].prev_hash = chain[i - 1].hash()
print("다시 꿴 체인 유효?", validate(chain) is None, "← 작업증명이 없으면 전체를 다시 잇는 데 블록당 해시 한 번이면 끝난다")

블록 수: 6 | 유효? True
  #0 prev=0000000000… hash=475e9da5b5… data='genesis'
  #1 prev=475e9da5b5… hash=e9668fba8f… data='A→B 1'
  #2 prev=e9668fba8f… hash=dcdec3cf8e… data='B→C 2'
  #3 prev=dcdec3cf8e… hash=daa1496cf4… data='C→D 3'
  #4 prev=daa1496cf4… hash=0e85befdef… data='D→E 4'
  #5 prev=0e85befdef… hash=3c35f3caa0… data='E→F 5'

--- 3번 블록의 거래를 '3' 에서 '300' 으로 바꾼다 ---
처음으로 끊어진 자리: 블록 # 4 (3번을 바꿨는데 4번이 틀렸다고 나온다 — 4번의 prev_hash 가 옛 해시)
  #3 prev=dcdec3cf8e… 실제 직전 해시=dcdec3cf8e…  일치=True
  #4 prev=daa1496cf4… 실제 직전 해시=a1b2cd168c…  일치=False
  #5 prev=0e85befdef… 실제 직전 해시=0e85befdef…  일치=True

--- 변조자가 이후 블록의 prev_hash 를 전부 다시 적는다 ---
다시 꿴 체인 유효? True ← 작업증명이 없으면 전체를 다시 잇는 데 블록당 해시 한 번이면 끝난다


In [ ]:
# NB07-2: 두 사슬이 다 「유효」할 때 — 해시만으로는 어느 쪽이 원본인지 알 수 없다
original = build_chain(datas)
forged = build_chain([d if i != 3 else "C→D 300" for i, d in enumerate(datas)])
print("원본 체인 유효?", validate(original) is None, "| 위조 체인 유효?", validate(forged) is None)
print("마지막 블록 해시 — 원본:", original[-1].hash()[:16], "… 위조:", forged[-1].hash()[:16], "…")
print("두 체인은 서로 다르지만 각각 내부적으로 완벽히 일관된다. 「어느 것이 진짜인가」는 해시가 답하지 못한다.")
print("이 질문에 답하려고 들어온 것이 8장의 작업증명(비용)과 「가장 많은 작업이 쌓인 체인」 규칙이다.")

원본 체인 유효? True | 위조 체인 유효? True
마지막 블록 해시 — 원본: 3c35f3caa070a93e … 위조: 63f530852e4eed45 …
두 체인은 서로 다르지만 각각 내부적으로 완벽히 일관된다. 「어느 것이 진짜인가」는 해시가 답하지 못한다.
이 질문에 답하려고 들어온 것이 8장의 작업증명(비용)과 「가장 많은 작업이 쌓인 체인」 규칙이다.
